## Day 16 — Causal setup: SMS -> No-show

#### Imports + paths (auto-find repo root and DB)

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import duckdb


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from joblib import dump

# --- find repo root by walking upwards until we see Day-11 ---
HERE = Path.cwd().resolve()
REPO_ROOT = None
for p in [HERE] + list(HERE.parents):
    if (p / "Day-11").exists():
        REPO_ROOT = p
        break

if REPO_ROOT is None:
    raise FileNotFoundError("Could not find repo root (folder containing Day-11). Run this notebook from inside the repo.")

DAY16_DIR = REPO_ROOT / "Day-16"
REPORTS = DAY16_DIR / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

DB_PATH = REPO_ROOT / "Day-11" / "data" / "warehouse" / "day11_noshow.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(f"DuckDB not found at: {DB_PATH}")

print("REPO_ROOT:", REPO_ROOT)
print("DB_PATH:", DB_PATH)
print("REPORTS:", REPORTS)


REPO_ROOT: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science
DB_PATH: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-11\data\warehouse\day11_noshow.duckdb
REPORTS: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\reports


### Connect to DuckDB + list tables

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)

tables = con.execute("SHOW TABLES").fetchdf()
print("Tables:")
display(tables)


Tables:


,name
0,bronze_appointments
1,gold_appointments_base
2,gold_appointments_features_v1
3,gold_appointments_features_v1_patient_split
4,silver_appointments
5,split_patient_v1


### Auto-pick the right table (the one that has sms_received + label)

In [3]:
table_names = tables["name"].tolist()

def table_has_cols(tname, needed=("sms_received", "label")):
    cols = con.execute(f"DESCRIBE {tname}").fetchdf()["column_name"].str.lower().tolist()
    return all(n in cols for n in needed)

candidates = [t for t in table_names if table_has_cols(t)]
print("Candidate tables with sms_received + label:", candidates)

if len(candidates) == 0:
    raise ValueError("No table contains BOTH sms_received and label. Check your warehouse tables.")
ANALYSIS_TABLE = candidates[0]
print("Using ANALYSIS_TABLE:", ANALYSIS_TABLE)

schema_df = con.execute(f"DESCRIBE {ANALYSIS_TABLE}").fetchdf()
display(schema_df)


Candidate tables with sms_received + label: ['gold_appointments_base', 'gold_appointments_features_v1', 'gold_appointments_features_v1_patient_split', 'silver_appointments']
Using ANALYSIS_TABLE: gold_appointments_base


,column_name,column_type,null,key,default,extra
0,appointment_id,BIGINT,YES,None,None,None
1,person_id,BIGINT,YES,None,None,None
2,scheduled_ts,TIMESTAMP,YES,None,None,None
3,appointment_ts,TIMESTAMP,YES,None,None,None
4,scheduled_date,DATE,YES,None,None,None
5,appointment_date,DATE,YES,None,None,None
6,lead_time_days,INTEGER,YES,None,None,None
7,date_inversion_flag,INTEGER,YES,None,None,None
8,age,DOUBLE,YES,None,None,None
9,gender,VARCHAR,YES,None,None,None


### Load analysis data + define A and Y + sanity checks

In [4]:
df = con.execute(f"SELECT * FROM {ANALYSIS_TABLE}").fetchdf()
print("Shape:", df.shape)

A_COL = "sms_received"
Y_COL = "label"          # will verify direction next

# Keep rows where A and Y exist
df = df.dropna(subset=[A_COL, Y_COL]).copy()

# Force to int 0/1 if possible
df[A_COL] = pd.to_numeric(df[A_COL], errors="coerce").astype("Int64")
df[Y_COL] = pd.to_numeric(df[Y_COL], errors="coerce").astype("Int64")

# Basic checks
print("A value counts:")
display(df[A_COL].value_counts(dropna=False))

print("Y value counts:")
display(df[Y_COL].value_counts(dropna=False))

# Confirm whether label=1 means no-show by checking association with sms_received
# (in this dataset, typically SMS reduces no-show, so we'd expect E[Y|A=1] < E[Y|A=0] if 1=no-show)
summary = df.groupby(A_COL)[Y_COL].agg(["mean","count"])
summary.columns = ["mean_Y","n"]
display(summary)


Shape: (110516, 18)
A value counts:


sms_received
0    75035
1    35481
Name: count, dtype: Int64

Y value counts:


label
0    88205
1    22311
Name: count, dtype: Int64

,mean_Y,n
sms_received,,
0,0.166949,75035
1,0.275753,35481


### Causal estimand + confounder set (write a small report header)

In [5]:
# We define Y=1 as "no-show" unless your sanity check suggests otherwise.
# Estimand: ATE = E[Y(1) - Y(0)] where A=1 means received SMS.

estimand_text = {
    "question": "What is the causal effect of receiving an SMS reminder on no-show?",
    "treatment_A": A_COL,
    "outcome_Y": Y_COL,
    "estimand": "ATE = E[Y(1) - Y(0)] on probability of no-show (Y=1)",
    "identification_assumptions": [
        "Consistency",
        "Conditional exchangeability given observed covariates X",
        "Positivity (overlap): 0 < P(A=1|X) < 1 for relevant X",
        "No interference / well-defined treatment"
    ]
}

with open(REPORTS / "DAY16_estimand.json", "w") as f:
    json.dump(estimand_text, f, indent=2)

estimand_text


{'question': 'What is the causal effect of receiving an SMS reminder on no-show?',
 'treatment_A': 'sms_received',
 'outcome_Y': 'label',
 'estimand': 'ATE = E[Y(1) - Y(0)] on probability of no-show (Y=1)',
 'identification_assumptions': ['Consistency',
  'Conditional exchangeability given observed covariates X',
  'Positivity (overlap): 0 < P(A=1|X) < 1 for relevant X',
  'No interference / well-defined treatment']}

### Choose covariates X (confounders) and prep modeling frame

In [6]:
# Confounders: variables that could affect BOTH SMS assignment and no-show.
# We'll exclude IDs and post-treatment variables (none obvious here besides A itself).

ID_COLS = ["appointment_id", "person_id"]

# Candidate covariates from your schema
numeric_cols = [
    "age", "lead_time_days", "lead_time_clipped", "lead_time_log1p",
    "sched_hour", "prior_appt_count", "nbhd_n"
]
categorical_cols = [
    "gender", "neighbourhood", "lead_time_bin"
]

# These are numeric but represent calendar categories; treat as categorical
calendar_as_cat = ["appt_dow", "appt_month", "sched_dow", "sched_month"]

# Binary-ish health indicators
binary_cols = [
    "scholarship", "hipertension", "diabetes", "alcoholism", "handcap"
]

X_cols = [c for c in (numeric_cols + categorical_cols + calendar_as_cat + binary_cols) if c in df.columns]

print("Using covariates X_cols:")
print(X_cols)

# modeling frame
model_df = df[[A_COL, Y_COL] + X_cols].copy()

# Drop rows missing A (required). Missing covariates allowed (imputed).
model_df = model_df.dropna(subset=[A_COL]).copy()

print("Modeling shape:", model_df.shape)


Using covariates X_cols:
['age', 'lead_time_days', 'gender', 'neighbourhood', 'scholarship', 'hipertension', 'diabetes', 'alcoholism', 'handcap']
Modeling shape: (110516, 11)


### Fit propensity model P(A=1|X)

In [7]:
X = model_df[X_cols]
A = model_df[A_COL].astype(int).to_numpy()

# Separate lists for preprocessing
num_use = [c for c in numeric_cols + binary_cols if c in X_cols]
cat_use = [c for c in categorical_cols + calendar_as_cat if c in X_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_use),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_use),
    ],
    remainder="drop"
)

prop_model = LogisticRegression(max_iter=2000, solver="lbfgs")

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", prop_model)
])

X_train, X_test, A_train, A_test = train_test_split(X, A, test_size=0.25, random_state=42, stratify=A)
pipe.fit(X_train, A_train)

ps_test = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(A_test, ps_test)
print("Propensity model ROC-AUC (held-out):", round(auc, 4))

# Propensity scores for all rows
ps = pipe.predict_proba(X)[:, 1]
model_df["pscore"] = ps
model_df["A"] = A
model_df["Y"] = model_df[Y_COL].astype(int)
model_df[["A","Y","pscore"]].head()


Propensity model ROC-AUC (held-out): 0.8051


,A,Y,pscore
0,0,0,0.226243
1,0,0,0.197466
2,0,0,0.289619
3,0,0,0.173862
4,0,0,0.218470


### Overlap / positivity diagnostics (the actual Day-16 requirement)

In [8]:
treated = model_df.loc[model_df["A"] == 1, "pscore"]
control = model_df.loc[model_df["A"] == 0, "pscore"]

overlap_stats = {
    "n": int(model_df.shape[0]),
    "treat_rate": float(model_df["A"].mean()),
    "ps_treated_min": float(treated.min()),
    "ps_treated_p01": float(np.quantile(treated, 0.01)),
    "ps_treated_p05": float(np.quantile(treated, 0.05)),
    "ps_treated_median": float(np.median(treated)),
    "ps_treated_p95": float(np.quantile(treated, 0.95)),
    "ps_treated_p99": float(np.quantile(treated, 0.99)),
    "ps_treated_max": float(treated.max()),
    "ps_control_min": float(control.min()),
    "ps_control_p01": float(np.quantile(control, 0.01)),
    "ps_control_p05": float(np.quantile(control, 0.05)),
    "ps_control_median": float(np.median(control)),
    "ps_control_p95": float(np.quantile(control, 0.95)),
    "ps_control_p99": float(np.quantile(control, 0.99)),
    "ps_control_max": float(control.max()),
}

with open(REPORTS / "DAY16_overlap_stats.json", "w") as f:
    json.dump(overlap_stats, f, indent=2)

overlap_stats


{'n': 110516,
 'treat_rate': 0.32104853595859423,
 'ps_treated_min': 0.09726632767977533,
 'ps_treated_p01': 0.14682425238878835,
 'ps_treated_p05': 0.19219636872301502,
 'ps_treated_median': 0.3892738741421624,
 'ps_treated_p95': 0.8749152001858937,
 'ps_treated_p99': 0.9720938043682307,
 'ps_treated_max': 0.9999835738608366,
 'ps_control_min': 0.06830104877980157,
 'ps_control_p01': 0.09845677949196265,
 'ps_control_p05': 0.11707580337204367,
 'ps_control_median': 0.20670008460514272,
 'ps_control_p95': 0.6443521661857363,
 'ps_control_p99': 0.9190723598023398,
 'ps_control_max': 0.9999780080883235}

### Plot overlap (save image)

In [9]:
plt.figure()
plt.hist(control, bins=30, alpha=0.6, label="A=0 (no SMS)")
plt.hist(treated, bins=30, alpha=0.6, label="A=1 (SMS)")
plt.xlabel("Propensity score P(A=1|X)")
plt.ylabel("Count")
plt.title("Positivity / Overlap Check (Propensity Distributions)")
plt.legend()
plt.savefig(REPORTS / "DAY16_pscore_overlap.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", REPORTS / "DAY16_pscore_overlap.png")


Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\reports\DAY16_pscore_overlap.png


### Simple positivity “red flag” rates + optional trimming thresholds

In [10]:
# How much mass is near 0 or 1?
eps = 0.01
near0 = float((model_df["pscore"] < eps).mean())
near1 = float((model_df["pscore"] > 1 - eps).mean())

print("Share pscore < 0.01:", round(near0, 4))
print("Share pscore > 0.99:", round(near1, 4))

trim_rule = {
    "eps": eps,
    "share_ps_lt_eps": near0,
    "share_ps_gt_1_minus_eps": near1,
    "recommended_trim": "If either share is large, consider trimming to [0.01, 0.99] before IPW/AIPW (Day 17)."
}

with open(REPORTS / "DAY16_positivity_flags.json", "w") as f:
    json.dump(trim_rule, f, indent=2)

trim_rule


Share pscore < 0.01: 0.0
Share pscore > 0.99: 0.002


{'eps': 0.01,
 'share_ps_lt_eps': 0.0,
 'share_ps_gt_1_minus_eps': 0.002017807376307503,
 'recommended_trim': 'If either share is large, consider trimming to [0.01, 0.99] before IPW/AIPW (Day 17).'}

### Save outputs for Day 17 (pscores + modeling frame)

In [11]:
out_path = REPORTS / "DAY16_model_frame_with_pscore.csv"
model_df.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Also saved overlap plot + json stats in:", REPORTS)


Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\reports\DAY16_model_frame_with_pscore.csv
Also saved overlap plot + json stats in: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\reports


### Write a short Day-16 “causal note” (blog-ready skeleton)

In [12]:
note = []
note.append("# Day 16 — Causal setup: SMS reminders → no-show\n")
note.append("## Estimand\n")
note.append("We estimate the average treatment effect (ATE) of receiving an SMS reminder on the probability of no-show.\n")
note.append(f"- Treatment A: `{A_COL}` (1 = received SMS)\n")
note.append(f"- Outcome Y: `{Y_COL}` (verify 1 = no-show; see sanity table)\n")
note.append("\n## Confounders X\n")
note.append("We adjust for pre-treatment variables that can affect both SMS assignment and attendance/no-show.\n")
note.append("Covariates used:\n")
note.append(", ".join(X_cols) + "\n")
note.append("\n## Propensity model and overlap\n")
note.append(f"- Propensity model: logistic regression with one-hot encoding for categorical variables\n")
note.append(f"- Held-out ROC-AUC: {auc:.4f}\n")
note.append("Overlap diagnostics were saved in `DAY16_overlap_stats.json` and `DAY16_pscore_overlap.png`.\n")
note.append("\n## Next (Day 17)\n")
note.append("Use propensity scores to estimate ATE via IPW and AIPW, with trimming if overlap is weak.\n")

md_path = REPORTS / "DAY16_causal_note.md"
md_path.write_text("\n".join(note), encoding="utf-8")
print("Saved:", md_path)


Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\reports\DAY16_causal_note.md


### Setup output folder (Day 16 reports)

In [13]:
from pathlib import Path

REPORTS = (Path("..").resolve() / "Day-16" / "reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print("Saving Day-16 outputs to:", REPORTS)


Saving Day-16 outputs to: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\Day-16\reports


### Quick sanity summary (and naïve difference)

In [14]:
import numpy as np
import pandas as pd

# model_df must already contain: A, Y, pscore
# from your previous cell:
# model_df["A"] = A
# model_df["Y"] = model_df[Y_COL].astype(int)
# model_df["pscore"] = ps

summary = model_df.groupby("A").agg(
    n=("Y", "size"),
    mean_Y=("Y", "mean"),
    mean_ps=("pscore", "mean"),
    p10_ps=("pscore", lambda s: np.quantile(s, 0.10)),
    p90_ps=("pscore", lambda s: np.quantile(s, 0.90)),
).reset_index()

naive_diff = float(summary.loc[summary["A"]==1, "mean_Y"].values[0] - summary.loc[summary["A"]==0, "mean_Y"].values[0])

print(summary)
print("\nNaive difference in mean(Y):", round(naive_diff, 6))

summary.to_csv(REPORTS / "DAY16_treatment_outcome_summary.csv", index=False)


   A      n    mean_Y   mean_ps    p10_ps    p90_ps
0  0  75035  0.166949  0.262389  0.135084  0.488990
1  1  35481  0.275753  0.445460  0.222315  0.754849

Naive difference in mean(Y): 0.108804


### Propensity score overlap diagnostics (tables)

In [15]:
def pscore_quantiles(df, col="pscore"):
    qs = [0.0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
    return df[col].quantile(qs).to_frame(name="pscore").reset_index().rename(columns={"index":"quantile"})

q0 = pscore_quantiles(model_df[model_df["A"]==0])
q1 = pscore_quantiles(model_df[model_df["A"]==1])

q0["A"] = 0
q1["A"] = 1

q = pd.concat([q0, q1], ignore_index=True)
q = q[["A","quantile","pscore"]]

print(q)

q.to_csv(REPORTS / "DAY16_pscore_quantiles_by_A.csv", index=False)

# Common support using min/max by group
min0, max0 = model_df.loc[model_df["A"]==0, "pscore"].min(), model_df.loc[model_df["A"]==0, "pscore"].max()
min1, max1 = model_df.loc[model_df["A"]==1, "pscore"].min(), model_df.loc[model_df["A"]==1, "pscore"].max()

lower = max(min0, min1)
upper = min(max0, max1)

print("\nControl pscore range:", (float(min0), float(max0)))
print("Treated pscore range:", (float(min1), float(max1)))
print("Common support interval:", (float(lower), float(upper)))

model_df["in_common_support"] = (model_df["pscore"] >= lower) & (model_df["pscore"] <= upper)
print("\n% outside common support:", round(100*(~model_df["in_common_support"]).mean(), 3))


    A  quantile    pscore
0   0      0.00  0.068301
1   0      0.01  0.098457
2   0      0.05  0.117076
3   0      0.10  0.135084
4   0      0.25  0.170649
5   0      0.50  0.206700
6   0      0.75  0.281022
7   0      0.90  0.488990
8   0      0.95  0.644352
9   0      0.99  0.919072
10  0      1.00  0.999978
11  1      0.00  0.097266
12  1      0.01  0.146824
13  1      0.05  0.192196
14  1      0.10  0.222315
15  1      0.25  0.277313
16  1      0.50  0.389274
17  1      0.75  0.589186
18  1      0.90  0.754849
19  1      0.95  0.874915
20  1      0.99  0.972094
21  1      1.00  0.999984

Control pscore range: (0.06830104877980157, 0.9999780080883235)
Treated pscore range: (0.09726632767977533, 0.9999835738608366)
Common support interval: (0.09726632767977533, 0.9999780080883235)

% outside common support: 0.473


### Propensity overlap plot (hist overlay)

In [16]:
import matplotlib.pyplot as plt

p0 = model_df.loc[model_df["A"]==0, "pscore"].to_numpy()
p1 = model_df.loc[model_df["A"]==1, "pscore"].to_numpy()

plt.figure()
plt.hist(p0, bins=50, alpha=0.6, density=True, label="A=0 (no SMS)")
plt.hist(p1, bins=50, alpha=0.6, density=True, label="A=1 (SMS)")
plt.axvline(lower, linestyle="--")
plt.axvline(upper, linestyle="--")
plt.xlabel("Propensity score P(A=1 | X)")
plt.ylabel("Density")
plt.title("Propensity Score Overlap (Common Support)")
plt.legend()
plt.savefig(REPORTS / "DAY16_pscore_overlap.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", REPORTS / "DAY16_pscore_overlap.png")


Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\Day-16\reports\DAY16_pscore_overlap.png


### Positivity “red flags” (extremes + trimming rate)

In [17]:
# Common “rule of thumb” positivity window (not mandatory, but informative)
lo, hi = 0.05, 0.95
outside_05_95 = ((model_df["pscore"] < lo) | (model_df["pscore"] > hi)).mean()

# Also check very extreme
outside_01_99 = ((model_df["pscore"] < 0.01) | (model_df["pscore"] > 0.99)).mean()

print("% pscore outside [0.05,0.95]:", round(100*outside_05_95, 3))
print("% pscore outside [0.01,0.99]:", round(100*outside_01_99, 3))

diag = pd.DataFrame({
    "metric": ["outside_05_95", "outside_01_99", "outside_common_support"],
    "value": [float(outside_05_95), float(outside_01_99), float((~model_df["in_common_support"]).mean())]
})
diag.to_csv(REPORTS / "DAY16_positivity_diagnostics.csv", index=False)
diag


% pscore outside [0.05,0.95]: 0.856
% pscore outside [0.01,0.99]: 0.202


,metric,value
0,outside_05_95,0.008560
1,outside_01_99,0.002018
2,outside_common_support,0.004732


### Build a trimmed analysis frame for Day 17

In [18]:
# Use common support trimming for stability in IPW/AIPW tomorrow
analysis_df = model_df.loc[model_df["in_common_support"]].copy()

print("Original n:", len(model_df))
print("Trimmed  n:", len(analysis_df))
print("Trimmed %:", round(100*(1 - len(analysis_df)/len(model_df)), 3))

# Save only the essential columns for causal estimation (plus IDs if you want)
keep_cols = [c for c in ["appointment_id","person_id","A","Y","pscore"] if c in analysis_df.columns]
analysis_df[keep_cols].to_csv(REPORTS / "DAY16_analysis_df_min.csv", index=False)

print("Saved:", REPORTS / "DAY16_analysis_df_min.csv")


Original n: 110516
Trimmed  n: 109993
Trimmed %: 0.473
Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-16\Day-16\reports\DAY16_analysis_df_min.csv
